In [1]:
%pip install pandas numpy matplotlib seaborn scikit-learn torch


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn, optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

In [5]:
from pathlib import Path

csv_candidates = (
    Path("data/ddos_csv_100000.csv"),
    Path("../../data/ddos_csv_100000.csv"),
)
csv_path = next((path for path in csv_candidates if path.is_file()), None)
if csv_path is None:
    raise FileNotFoundError(f"CSV not found. Checked: {csv_candidates}")

df = pd.read_csv(csv_path)
df.head(5)

,Src Port,Dst Port,Protocol,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,Total Length of Bwd Packet,Fwd Packet Length Max,Fwd Packet Length Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,4425,49153,6,6434,3,4,196.0,193.0,196.0,0.0,...,32,0,0,0,0,1.474553e+15,1846.139934,1.474553e+15,1.474553e+15,1
1,4425,49153,6,1173,2,0,0.0,0.0,0.0,0.0,...,32,0,0,0,0,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,1
2,4426,49153,6,6498,3,4,198.0,193.0,198.0,0.0,...,32,0,0,0,0,1.474553e+15,1817.796927,1.474553e+15,1.474553e+15,1
3,4426,49153,6,1217,2,0,0.0,0.0,0.0,0.0,...,32,0,0,0,0,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,1
4,4977,49152,6,9928,3,4,186.0,1170.0,186.0,0.0,...,32,0,0,0,0,1.474553e+15,2305.375003,1.474553e+15,1.474553e+15,1


In [6]:
n_rows, n_columns = df.shape
print(f"Rows: {n_rows:,}")
print(f"Columns: {n_columns:,}")
print(f"Duplicate rows: {df.duplicated().sum():,}")

missing = df.isna().sum()
missing_summary = pd.DataFrame({
    "missing": missing,
    "missing_pct": 100 * missing / len(df),
})
missing_summary = missing_summary[missing_summary["missing"] > 0].sort_values(
    "missing", ascending=False
)

print("\nMissing values:")
if missing_summary.empty:
    print("No NaN values found.")
else:
    display(missing_summary)

numeric = df.select_dtypes(include=np.number)
infinite_counts = np.isinf(numeric).sum()
infinite_counts = infinite_counts[infinite_counts > 0]

print("\nInfinite values:")
if infinite_counts.empty:
    print("No infinite values found.")
else:
    display(infinite_counts.rename("infinite_count").to_frame())

print("\nNumeric descriptive statistics:")
stats = numeric.replace([np.inf, -np.inf], np.nan).describe().T
display(stats[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]])

if "Label" in df.columns:
    print("\nLabel distribution:")
    label_counts = df["Label"].value_counts(dropna=False)
    display(pd.DataFrame({
        "count": label_counts,
        "percentage": 100 * label_counts / len(df),
    }))

Rows: 100,000
Columns: 80
Duplicate rows: 661

Missing values:
No NaN values found.

Infinite values:
No infinite values found.

Numeric descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
Src Port,100000.0,2.777612e+04,2.157176e+04,0.0,4.372000e+03,2.793300e+04,4.915200e+04,6.553500e+04
Dst Port,100000.0,1.027072e+04,1.920876e+04,0.0,8.000000e+01,8.000000e+01,4.324250e+03,6.552600e+04
Protocol,100000.0,7.217150e+00,4.061569e+00,0.0,6.000000e+00,6.000000e+00,6.000000e+00,1.700000e+01
Flow Duration,100000.0,2.212085e+07,3.144865e+07,27.0,1.395122e+05,1.495958e+07,2.032164e+07,1.200000e+08
Total Fwd Packet,100000.0,6.148800e+00,1.945326e+01,1.0,3.000000e+00,4.000000e+00,5.000000e+00,2.790000e+03
...,...,...,...,...,...,...,...,...
Idle Mean,100000.0,1.117685e+15,5.175939e+14,0.0,7.640479e+14,1.474563e+15,1.474657e+15,1.528096e+15
Idle Std,100000.0,2.924960e+14,4.255766e+14,0.0,4.424508e+03,6.832233e+06,7.373334e+14,1.080527e+15
Idle Max,100000.0,1.292596e+15,5.254731e+14,0.0,1.474592e+15,1.474681e+15,1.528096e+15,1.528096e+15
Idle Min,100000.0,7.870662e+14,7.464156e+14,0.0,9.116082e+06,1.474563e+15,1.474657e+15,1.528096e+15



Label distribution:


,count,percentage
Label,,
1,50000,50.0
0,50000,50.0


In [ ]:
from time import perf_counter

# Remove exact duplicates before splitting to reduce train/test leakage.
model_df = df.drop_duplicates().copy()
X = model_df.drop(columns="Label")
y = model_df["Label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Use the same stratified test sample for every model.
X_test_eval, _, y_test_eval, _ = train_test_split(
    X_test,
    y_test,
    train_size=min(5_000, len(X_test)),
    random_state=42,
    stratify=y_test,
)

models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=300, class_weight="balanced")
    ),
    "Random Forest": make_pipeline(
        RandomForestClassifier(
            n_estimators=80,
            n_jobs=-1,
            class_weight="balanced_subsample",
            random_state=42,
        )
    ),
    "Hist Gradient Boosting": make_pipeline(
        HistGradientBoostingClassifier(max_iter=100, random_state=42)
    ),
    "KNN (10k train rows)": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    ),
}

slow_model_train_limits = {
    "KNN (10k train rows)": 10_000,
}
results = []
predictions = {}

for name, model in models.items():
    train_limit = slow_model_train_limits.get(name)
    if train_limit is not None:
        X_fit, _, y_fit, _ = train_test_split(
            X_train,
            y_train,
            train_size=min(train_limit, len(X_train)),
            random_state=42,
            stratify=y_train,
        )
    else:
        X_fit, y_fit = X_train, y_train

    started = perf_counter()
    model.fit(X_fit, y_fit)
    y_pred = model.predict(X_test_eval)
    elapsed = perf_counter() - started
    predictions[name] = y_pred

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test_eval)[:, 1]
    else:
        y_score = model.decision_function(X_test_eval)

    results.append({
        "model": name,
        "train_rows": len(X_fit),
        "test_rows": len(X_test_eval),
        "fit_predict_seconds": elapsed,
        "accuracy": accuracy_score(y_test_eval, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test_eval, y_pred),
        "precision": precision_score(y_test_eval, y_pred, zero_division=0),
        "recall": recall_score(y_test_eval, y_pred, zero_division=0),
        "f1": f1_score(y_test_eval, y_pred, zero_division=0),
        "macro_f1": f1_score(y_test_eval, y_pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_test_eval, y_pred),
        "cohen_kappa": cohen_kappa_score(y_test_eval, y_pred),
        "roc_auc": roc_auc_score(y_test_eval, y_score),
    })

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
print(f"Rows after removing duplicates: {len(model_df):,}")
print(f"Training rows available: {len(X_train):,}; evaluation rows: {len(X_test_eval):,}")
display(results_df.round(4))



Rows after removing duplicates: 99,339
Training rows available: 79,471; evaluation rows: 5,000


,model,train_rows,test_rows,fit_predict_seconds,accuracy,balanced_accuracy,precision,recall,f1,macro_f1,mcc,cohen_kappa,roc_auc
0,Logistic Regression,79471,5000,0.2714,1.000,1.000,1.0000,1.0000,1.000,1.000,1.000,1.000,1.0000
1,Random Forest,79471,5000,0.6704,1.000,1.000,1.0000,1.0000,1.000,1.000,1.000,1.000,1.0000
2,Hist Gradient Boosting,79471,5000,1.0231,1.000,1.000,1.0000,1.0000,1.000,1.000,1.000,1.000,1.0000
3,KNN (10k train rows),10000,5000,0.0901,0.999,0.999,0.9996,0.9984,0.999,0.999,0.998,0.998,0.9996


In [9]:
from time import perf_counter
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recreate the sklearn split so both approaches use the same test rows.
nn_df = df.drop_duplicates().copy()
X_nn = nn_df.drop(columns="Label")
y_nn = nn_df["Label"].astype(int)
X_train_nn, X_test_nn, y_train_nn, y_test_nn = train_test_split(
    X_nn, y_nn, test_size=0.2, random_state=42, stratify=y_nn
)
X_test_eval_nn, _, y_test_eval_nn, _ = train_test_split(
    X_test_nn,
    y_test_nn,
    train_size=min(5_000, len(X_test_nn)),
    random_state=42,
    stratify=y_test_nn,
)

# Fit scaling only on the training data.
scaler_nn = StandardScaler()
X_train_scaled = scaler_nn.fit_transform(X_train_nn).astype(np.float32)
X_test_scaled = scaler_nn.transform(X_test_eval_nn).astype(np.float32)
y_train_array = y_train_nn.to_numpy(dtype=np.float32).reshape(-1, 1)

train_dataset = TensorDataset(
    torch.from_numpy(X_train_scaled),
    torch.from_numpy(y_train_array),
)
train_loader = DataLoader(
    train_dataset, batch_size=1_024, shuffle=True, num_workers=0
)

nn_model = nn.Sequential(
    nn.Linear(X_train_scaled.shape[1], 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(nn_model.parameters(), lr=1e-3, weight_decay=1e-4)

epochs = 15
started = perf_counter()
for epoch in range(epochs):
    nn_model.train()
    total_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = nn_model(batch_X)
        loss = loss_fn(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_X)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - loss: {total_loss / len(train_dataset):.4f}")

nn_model.eval()
with torch.no_grad():
    test_X_tensor = torch.from_numpy(X_test_scaled).to(device)
    y_score_nn = torch.sigmoid(nn_model(test_X_tensor)).cpu().numpy().ravel()
nn_seconds = perf_counter() - started
y_pred_nn = (y_score_nn >= 0.5).astype(int)

nn_metrics = {
    "model": "PyTorch MLP",
    "train_rows": len(X_train_nn),
    "test_rows": len(y_test_eval_nn),
    "fit_predict_seconds": nn_seconds,
    "accuracy": accuracy_score(y_test_eval_nn, y_pred_nn),
    "balanced_accuracy": balanced_accuracy_score(y_test_eval_nn, y_pred_nn),
    "precision": precision_score(y_test_eval_nn, y_pred_nn, zero_division=0),
    "recall": recall_score(y_test_eval_nn, y_pred_nn, zero_division=0),
    "f1": f1_score(y_test_eval_nn, y_pred_nn, zero_division=0),
    "macro_f1": f1_score(y_test_eval_nn, y_pred_nn, average="macro", zero_division=0),
    "mcc": matthews_corrcoef(y_test_eval_nn, y_pred_nn),
    "cohen_kappa": cohen_kappa_score(y_test_eval_nn, y_pred_nn),
    "roc_auc": roc_auc_score(y_test_eval_nn, y_score_nn),
}
nn_metrics_df = pd.DataFrame([nn_metrics])
display(nn_metrics_df.round(4))

print("Classification report - PyTorch MLP")
print(classification_report(y_test_eval_nn, y_pred_nn, digits=4))
print("Confusion matrix - rows=actual, columns=predicted")
display(pd.DataFrame(
    confusion_matrix(y_test_eval_nn, y_pred_nn),
    index=["actual 0", "actual 1"],
    columns=["predicted 0", "predicted 1"],
))

Epoch 5/15 - loss: 0.0009
Epoch 10/15 - loss: 0.0004
Epoch 15/15 - loss: 0.0003


,model,train_rows,test_rows,fit_predict_seconds,accuracy,balanced_accuracy,precision,recall,f1,macro_f1,mcc,cohen_kappa,roc_auc
0,PyTorch MLP,79471,5000,7.5998,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


Classification report - PyTorch MLP
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000      2517
           1     1.0000    1.0000    1.0000      2483

    accuracy                         1.0000      5000
   macro avg     1.0000    1.0000    1.0000      5000
weighted avg     1.0000    1.0000    1.0000      5000

Confusion matrix - rows=actual, columns=predicted


,predicted 0,predicted 1
actual 0,2517,0
actual 1,0,2483
